# 🎵 Waveform Studio — Audio Visualizer Video Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mnchrmXD/waveform-studio/blob/main/waveform_studio.ipynb)

Generate studio-grade, audio-reactive visualizer videos directly in Google Colab using **Waveform Studio** (`mnchrmXD/waveform-studio`).

### Quick Flow:
1. **Launch Server**: Run Cell 1 to start the high-speed Node.js + FFmpeg headless rendering engine on `http://localhost:3000`.
2. **Upload Payload**: Run Cell 2 to upload your `payload.json` exported from Waveform Studio (or skip to use built-in defaults).
3. **Render Video**: Run Cell 3 to encode with hardware acceleration and stream the video.
4. **Preview & Download**: Run Cells 4 & 5 to preview in-notebook and download.

## 1. Setup Environment & Launch Server
Prepares dependencies and launches the Waveform Studio headless rendering engine in Colab in the background on `http://localhost:3000`.

In [ ]:
# Install Python utilities
!pip install -q requests tqdm

import os
import sys
import time
import json
import subprocess
import requests
import threading
from tqdm import tqdm
from IPython.display import HTML, Video, display

# 1. Proactive Colab GPU & NVENC Verification & Binary Provisioning
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
has_gpu = False
gpu_name = None

try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False
    )
    if smi.returncode == 0 and smi.stdout.strip():
        gpu_name = smi.stdout.strip().split('\n')[0]
        has_gpu = True
        print(f"🔥 Google Colab GPU Detected: {gpu_name}")
except Exception:
    pass

ffmpeg_bin = '/usr/local/bin/ffmpeg'
has_nvenc = False

if has_gpu:
    # Check if any existing ffmpeg binary supports h264_nvenc
    for candidate in ['/usr/local/bin/ffmpeg', '/usr/bin/ffmpeg', 'ffmpeg']:
        try:
            chk = subprocess.run([candidate, '-encoders'], capture_output=True, text=True, check=False)
            if 'h264_nvenc' in (chk.stdout or ''):
                has_nvenc = True
                ffmpeg_bin = candidate
                break
        except Exception:
            pass

    # If system ffmpeg lacks NVENC (standard Ubuntu Colab packages lack non-free nvenc headers),
    # download official pre-built Linux64 FFmpeg with NVENC & CUDA support (~3s in Colab datacenter network)
    if not has_nvenc:
        print("⚡ Standard Colab ffmpeg lacks NVENC support. Provisioning NVENC-enabled FFmpeg binary...")
        try:
            subprocess.run(
                "curl -sL https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz | tar -xJ --strip-components=2 -C /usr/local/bin --wildcards '*/bin/ffmpeg'",
                shell=True, check=False
            )
            if os.path.exists('/usr/local/bin/ffmpeg'):
                chk2 = subprocess.run(['/usr/local/bin/ffmpeg', '-encoders'], capture_output=True, text=True, check=False)
                if 'h264_nvenc' in (chk2.stdout or ''):
                    has_nvenc = True
                    ffmpeg_bin = '/usr/local/bin/ffmpeg'
                    print("✅ NVENC-enabled FFmpeg successfully installed to /usr/local/bin/ffmpeg!")
        except Exception as e:
            print(f"Notice while provisioning NVENC binary: {e}")

    if has_nvenc:
        os.environ["FFMPEG_PATH"] = ffmpeg_bin
        print(f"⚡ Hardware NVENC Acceleration: ACTIVE ({ffmpeg_bin})")
    else:
        print("ℹ️ Notice: Hardware NVENC binary unavailable. Server will use multi-threaded CPU libx264.")
else:
    print("ℹ️ Running on CPU runtime.")
    if IN_COLAB:
        print("💡 TIP: For 10-20x faster export, switch Colab runtime: Runtime > Change runtime type > T4 GPU.")

# 2. Ensure repository is available and active directory
if IN_COLAB:
    if not os.path.exists('server.ts') and not os.path.exists('waveform-studio'):
        print("📥 Cloning Waveform Studio repository...")
        !git clone --depth 1 https://github.com/mnchrmXD/waveform-studio.git
        %cd waveform-studio
    elif os.path.exists('waveform-studio') and not os.path.exists('server.ts'):
        %cd waveform-studio

# 3. Fast install Node.js dependencies if not already present
if not os.path.exists('node_modules'):
    print("⚡ Installing Node.js & FFmpeg rendering dependencies...")
    !npm install --prefer-offline --no-audit --no-fund --loglevel=error

# 4. Check if server is already running on port 3000
API_URL = "http://localhost:3000"
server_ready = False

try:
    health_resp = requests.get(f"{API_URL}/api/health", timeout=2)
    if health_resp.status_code == 200:
        server_ready = True
        print("✅ Waveform Studio server is already running!")
except Exception:
    pass

# 5. Launch headless server in background (HEADLESS_ONLY skips Vite frontend for instant start)
if not server_ready:
    print("🚀 Starting Waveform Studio headless rendering engine...")
    env = os.environ.copy()
    env["HEADLESS_ONLY"] = "true"
    env["NODE_ENV"] = "production"
    if os.path.exists(ffmpeg_bin):
        env["FFMPEG_PATH"] = ffmpeg_bin
    
    server_proc = subprocess.Popen(
        ["npx", "tsx", "server.ts"],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    
    # Poll until server is ready
    start_wait = time.time()
    for _ in range(35):
        try:
            health_resp = requests.get(f"{API_URL}/api/health", timeout=1)
            if health_resp.status_code == 200:
                server_ready = True
                hw_resp = requests.get(f"{API_URL}/api/hardware-environment", timeout=1)
                hw_data = hw_resp.json() if hw_resp.status_code == 200 else {}
                gpu_meta = hw_data.get('gpu', {})
                enc = gpu_meta.get('activeEncoder', 'libx264')
                model = gpu_meta.get('model') or 'CPU'
                print(f"✅ Waveform Studio server ready in {time.time() - start_wait:.1f}s on {API_URL}!")
                print(f"   • Active Encoder: {enc} ({model})")
                print(f"   • Status: {hw_data.get('recommendation', '')}")
                break
        except Exception:
            time.sleep(0.5)

    if not server_ready:
        print("⚠️ Server initialization taking longer than expected. Check server output if needed.")

## 2. Upload Payload (JSON)
Upload your `payload.json` exported from Waveform Studio's **Payload Generator**.
*(If you skip this or don't upload a file, the renderer will automatically use default settings.)*

In [ ]:
payload = {}

try:
    from google.colab import files
    print("📤 Upload your payload.json (or cancel/skip to use default settings):")
    uploaded = files.upload()
    for fn, content in uploaded.items():
        if fn.endswith('.json'):
            try:
                payload = json.loads(content.decode('utf-8'))
                with open('payload.json', 'w') as f:
                    json.dump(payload, f, indent=2)
                print(f"✅ Successfully loaded and set payload from '{fn}'!")
                break
            except Exception as err:
                print(f"⚠️ Error parsing uploaded JSON: {err}")
except ImportError:
    # Running outside Colab
    if os.path.exists('payload.json'):
        with open('payload.json', 'r') as f:
            payload = json.load(f)
        print("✅ Loaded existing payload.json from workspace.")

# Fallback check for local payload.json if not yet loaded
if not payload and os.path.exists('payload.json'):
    try:
        with open('payload.json', 'r') as f:
            payload = json.load(f)
        print("✅ Loaded payload from local payload.json!")
    except Exception:
        pass

if payload:
    w = payload.get('video', {}).get('width', 1280)
    h = payload.get('video', {}).get('height', 720)
    fps = payload.get('video', {}).get('fps', 30)
    fmt = payload.get('video', {}).get('format', 'mp4')
    style = payload.get('settings', {}).get('style', 'default')
    print(f"🎯 Payload ready: {style} style, {w}×{h} @ {fps}fps ({fmt})")
else:
    print("ℹ️ No JSON uploaded — renderer will use its built-in default settings.")

## 3. Render Video via Headless API
Submits the payload to `/api/render-video`. The server analyzes the audio, renders frames with hardware acceleration, encodes them with FFmpeg, and streams the video back in real-time.

In [ ]:
# 1. Ensure payload is loaded
if 'payload' not in globals() or payload is None:
    if os.path.exists('payload.json'):
        with open('payload.json', 'r') as f:
            payload = json.load(f)
    else:
        payload = {}

video_format = payload.get('video', {}).get('format', 'mp4') if isinstance(payload, dict) else 'mp4'
OUTPUT_FILENAME = f"waveform_render.{video_format}"

# 2. Inspect Hardware Acceleration Status
try:
    hw_info = requests.get(f"{API_URL}/api/hardware-environment", timeout=2).json()
    gpu_meta = hw_info.get('gpu', {})
    enc_name = gpu_meta.get('activeEncoder', 'libx264')
    if gpu_meta.get('supportsNvenc'):
        hw_tag = f"NVIDIA {gpu_meta.get('model', 'GPU')} (NVENC)"
    else:
        hw_tag = f"CPU ({enc_name})"
except Exception:
    hw_tag = "Headless Pipeline"

job_id = f"colab_{int(time.time())}"
payload['jobId'] = job_id
render_url = f"{API_URL}/api/render-video?jobId={job_id}"

print(f"🚀 Submitting render request: {job_id}")
print(f"⚡ Active Hardware: [{hw_tag}]")

# 3. Execute Render in background thread while tracking live progress
render_result = [None]
render_err = [None]

def perform_render():
    try:
        res = requests.post(render_url, json=payload, stream=True, timeout=600)
        render_result[0] = res
    except Exception as ex:
        render_err[0] = str(ex)

t = threading.Thread(target=perform_render)
t.daemon = True
t.start()

start_time = time.time()
last_pct = 0

# 4. Real-time Progress Bar (Frames, Speed in FPS, and ETA)
with tqdm(
    total=100,
    desc=f"🎨 Rendering Video [{hw_tag}]",
    unit="%",
    bar_format="{l_bar}{bar}| {n:.0f}% [{elapsed}<{remaining}, {rate_fmt}] {postfix}",
    dynamic_ncols=True
) as pbar:
    while t.is_alive():
        try:
            st = requests.get(f"{API_URL}/api/render-status/{job_id}", timeout=0.8)
            if st.status_code == 200:
                info = st.json()
                pct = int(info.get('progress', 0))
                if pct > last_pct:
                    pbar.update(pct - last_pct)
                    last_pct = pct
                
                cur_f = info.get('currentFrame')
                tot_f = info.get('totalFrames')
                speed_fps = info.get('fps')
                postfix = {}
                if cur_f is not None and tot_f is not None:
                    postfix['frames'] = f"{cur_f}/{tot_f}"
                if speed_fps:
                    postfix['speed'] = f"{speed_fps}fps"
                if postfix:
                    pbar.set_postfix(postfix)
                
                if info.get('status') == 'completed':
                    break
        except Exception:
            pass
        time.sleep(0.15)
    
    t.join()
    if last_pct < 100:
        pbar.update(100 - last_pct)

# 5. Save Output Video File
resp = render_result[0]
if resp and resp.status_code == 200:
    block_size = 1024 * 1024
    with open(OUTPUT_FILENAME, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=block_size):
            if chunk:
                f.write(chunk)
                
    elapsed = time.time() - start_time
    file_size_mb = os.path.getsize(OUTPUT_FILENAME) / (1024 * 1024)
    print(f"\n🎉 Render successfully completed in {elapsed:.1f}s!")
    print(f"📁 Output saved to: {OUTPUT_FILENAME} ({file_size_mb:.2f} MB)")
else:
    err_text = render_err[0] or (resp.text if resp else "Unknown error")
    print(f"\n❌ Render failed: {err_text}")

## 4. Preview & Playback Video
Preview the generated video directly within Colab.

In [ ]:
if os.path.exists(OUTPUT_FILENAME):
    print("Previewing rendered video:")
    display(Video(OUTPUT_FILENAME, embed=True, width=720))
else:
    print(f"File {OUTPUT_FILENAME} does not exist yet. Run Section 3 first.")

## 5. Download Video to Local Machine
Download the finished video directly to your computer.

In [ ]:
try:
    from google.colab import files
    if os.path.exists(OUTPUT_FILENAME):
        print(f"Downloading {OUTPUT_FILENAME}...")
        files.download(OUTPUT_FILENAME)
    else:
        print(f"File {OUTPUT_FILENAME} not found.")
except ImportError:
    print(f"Not running in Google Colab. File is saved at: {os.path.abspath(OUTPUT_FILENAME)}")